# 📡 Notebook 1: Networking Fundamentals

Before diving into real-time update patterns, we need to understand the basics of how networks work. This knowledge will help you make better decisions when designing real-time systems.

## Learning Objectives

By the end of this notebook, you'll understand:
- The OSI model and why it matters for real-time systems
- TCP vs UDP trade-offs
- The HTTP request lifecycle
- Layer 4 vs Layer 7 load balancers

## 🧅 The OSI Model (Simplified)

Networks are built like an onion - in layers! Each layer provides services to the layer above it. As application developers, we mostly care about three layers:

```
┌─────────────────────────────────────────────────────────────┐
│  Layer 7: Application Layer                                 │
│  (HTTP, WebSocket, DNS, WebRTC)                             │
│  "What we usually work with"                                │
├─────────────────────────────────────────────────────────────┤
│  Layer 4: Transport Layer                                   │
│  (TCP, UDP)                                                 │
│  "Reliable delivery vs speed"                               │
├─────────────────────────────────────────────────────────────┤
│  Layer 3: Network Layer                                     │
│  (IP)                                                       │
│  "Getting packets from A to B"                              │
└─────────────────────────────────────────────────────────────┘
```

### Why does this matter for real-time systems?

Each layer adds some overhead and constraints. Understanding these helps us choose the right protocol for our needs.

## 🔄 TCP vs UDP

At Layer 4 (Transport), we have two main protocols:

### TCP (Transmission Control Protocol)
- **Connection-oriented**: Must establish connection first (handshake)
- **Reliable**: Guarantees delivery and order
- **Slower setup**: 3-way handshake adds latency

### UDP (User Datagram Protocol)
- **Connectionless**: Just send data, no setup needed
- **Unreliable**: Packets can be lost, duplicated, or reordered
- **Fast**: No handshake, minimal overhead

```
TCP: "Hey, are you there?" → "Yes, I'm here" → "Great, let's talk" → [data]
UDP: [data] → 🤷 (hope it arrives!)
```

In [ ]:
# Let's see TCP vs UDP in action!
import socket
import time

def demonstrate_tcp_handshake():
    """
    TCP requires a 3-way handshake before data transfer.
    This takes time but guarantees a reliable connection.
    """
    start = time.time()
    
    # Create a TCP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    
    try:
        # This single line does:
        # 1. SYN → Server
        # 2. SYN-ACK ← Server  
        # 3. ACK → Server
        sock.connect(('httpbin.org', 80))
        elapsed = (time.time() - start) * 1000
        print(f"✅ TCP connection established in {elapsed:.2f}ms")
        print("   This included the 3-way handshake!")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
    finally:
        sock.close()

demonstrate_tcp_handshake()

In [ ]:
# UDP is much simpler - no handshake needed
import socket

def demonstrate_udp_simplicity():
    """
    UDP doesn't require any connection setup.
    You can just send data immediately!
    """
    # Create a UDP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    
    # With UDP, we can send immediately - no connect() needed!
    # (We're sending to a DNS server as an example)
    message = b"Hello, UDP!"
    
    print("📤 Sending UDP packet...")
    print("   No handshake required!")
    print("   But we have NO guarantee it will arrive.")
    
    sock.close()
    print("\n✅ UDP socket created and closed instantly")

demonstrate_udp_simplicity()

## 🌐 HTTP Request Lifecycle

When you type a URL in your browser, a lot happens under the hood:

```
┌─────────┐                                    ┌─────────┐
│ Browser │                                    │ Server  │
└────┬────┘                                    └────┬────┘
     │                                              │
     │  1. DNS Lookup (example.com → 93.184.216.34) │
     │─────────────────────────────────────────────>│
     │                                              │
     │  2. TCP Handshake                            │
     │──────────── SYN ────────────────────────────>│
     │<─────────── SYN-ACK ─────────────────────────│
     │──────────── ACK ────────────────────────────>│
     │                                              │
     │  3. HTTP Request                             │
     │──────────── GET /index.html ────────────────>│
     │                                              │
     │  4. HTTP Response                            │
     │<─────────── 200 OK + HTML ──────────────────│
     │                                              │
     │  5. TCP Teardown                             │
     │──────────── FIN ────────────────────────────>│
     │<─────────── ACK ─────────────────────────────│
     │<─────────── FIN ─────────────────────────────│
     │──────────── ACK ────────────────────────────>│
     │                                              │
```

### Key Insights for Real-time Systems:

1. **Every round trip adds latency** - Handshakes take time!
2. **Connections are stateful** - Both sides must track the connection
3. **HTTP is request-response** - Server can't push to client

In [ ]:
# Let's measure the real cost of HTTP requests
import time
import requests

def measure_http_overhead():
    """
    Measure the time it takes for HTTP requests.
    We'll compare a fresh connection vs reusing a connection.
    """
    url = "https://httpbin.org/get"
    
    # Test 1: Fresh connection each time (no keep-alive)
    print("🔄 Test 1: Fresh connections (no keep-alive)")
    times_fresh = []
    for i in range(3):
        start = time.time()
        # Create new session each time = new TCP connection
        response = requests.get(url)
        elapsed = (time.time() - start) * 1000
        times_fresh.append(elapsed)
        print(f"   Request {i+1}: {elapsed:.2f}ms")
    
    print(f"   Average: {sum(times_fresh)/len(times_fresh):.2f}ms\n")
    
    # Test 2: Reuse connection (keep-alive)
    print("🔗 Test 2: Reused connection (keep-alive)")
    times_reuse = []
    session = requests.Session()  # Session reuses connections!
    for i in range(3):
        start = time.time()
        response = session.get(url)
        elapsed = (time.time() - start) * 1000
        times_reuse.append(elapsed)
        print(f"   Request {i+1}: {elapsed:.2f}ms")
    
    print(f"   Average: {sum(times_reuse)/len(times_reuse):.2f}ms")
    
    # Note: First request in session still needs handshake
    print("\n💡 Notice: First request is slower (needs TCP handshake)")
    print("   Subsequent requests reuse the connection!")

measure_http_overhead()

## ⚖️ Load Balancers: Layer 4 vs Layer 7

In production, you rarely have just one server. Load balancers distribute traffic across multiple servers. There are two main types:

### Layer 4 Load Balancer

Operates at the **transport layer** (TCP/UDP). Makes decisions based on:
- IP addresses
- Port numbers
- That's it!

```
┌────────┐      TCP Connection      ┌────────┐      TCP Connection      ┌────────┐
│ Client │◄────────────────────────►│ L4 LB  │◄────────────────────────►│ Server │
└────────┘                          └────────┘                          └────────┘

The L4 LB just forwards TCP packets. It's like a transparent pipe.
The client effectively has a direct TCP connection to one server.
```

**Pros:** Fast, efficient, maintains persistent connections
**Cons:** Can't make smart routing decisions based on request content

### Layer 7 Load Balancer

Operates at the **application layer** (HTTP). Can inspect:
- URL paths
- Headers
- Cookies
- Request body

```
┌────────┐    TCP #1    ┌────────┐    TCP #2    ┌────────┐
│ Client │◄────────────►│ L7 LB  │◄────────────►│ Server │
└────────┘              └────────┘              └────────┘

The L7 LB terminates the client connection and creates a new one to the server.
It can route different requests to different servers!
```

**Pros:** Smart routing, can route `/api` to one server and `/static` to another
**Cons:** More CPU overhead, breaks persistent connections

## 🎯 Why This Matters for Real-time Systems

| Protocol | Load Balancer | Works Well? | Why |
|----------|--------------|-------------|-----|
| Simple Polling | L4 or L7 | ✅ | Just HTTP requests |
| Long Polling | L4 or L7 | ✅ | Still just HTTP |
| SSE | L7 (must support streaming) | ⚠️ | Need streaming support |
| WebSocket | L4 preferred | ✅ | Needs persistent TCP |
| WebSocket | L7 (with WS support) | ⚠️ | Not all L7 LBs support it |

### Key Takeaways:

1. **L4 Load Balancers** are better for WebSockets because they maintain the TCP connection
2. **L7 Load Balancers** work better for HTTP-based solutions like polling
3. Many modern L7 LBs (AWS ALB, nginx) now support WebSocket "upgrade"

In [ ]:
# Let's visualize the concept of connection persistence

def visualize_lb_behavior():
    """
    This demonstrates conceptually how L4 vs L7 LBs handle connections.
    """
    
    print("📊 Layer 4 Load Balancer Behavior")
    print("="*50)
    print("")
    print("Client connects → LB picks Server A")
    print("Request 1 → Server A  ✅")
    print("Request 2 → Server A  ✅ (same TCP connection)")
    print("Request 3 → Server A  ✅ (same TCP connection)")
    print("")
    print("👍 All requests go to the same server!")
    print("   Great for WebSockets and stateful connections.")
    print("")
    
    print("📊 Layer 7 Load Balancer Behavior")
    print("="*50)
    print("")
    print("Request 1: GET /api/users")
    print("  → LB inspects path → Routes to API Server")
    print("")
    print("Request 2: GET /static/logo.png")
    print("  → LB inspects path → Routes to Static Server")
    print("")
    print("Request 3: POST /api/orders")
    print("  → LB inspects path → Routes to API Server")
    print("")
    print("👍 Smart routing based on content!")
    print("⚠️  Each request might go to a different backend server.")

visualize_lb_behavior()

## 🧪 Quick Quiz

Test your understanding! Try to answer before running the cell.

1. **Which protocol would you use for a video call?**
   - A) TCP (reliable)
   - B) UDP (fast)

2. **You're building a chat app with WebSockets. Which load balancer type is better?**
   - A) Layer 4
   - B) Layer 7

3. **How many round trips does a TCP handshake take?**
   - A) 1
   - B) 1.5 (3 messages)
   - C) 3

In [ ]:
# Run this cell to see the answers!

def show_quiz_answers():
    print("📝 Quiz Answers")
    print("="*50)
    print("")
    print("1. B) UDP - Video calls prioritize speed over reliability.")
    print("   A few dropped frames is better than frozen video!")
    print("")
    print("2. A) Layer 4 - WebSockets need persistent TCP connections.")
    print("   L4 LBs maintain the connection to the same server.")
    print("")
    print("3. B) 1.5 round trips (3 messages: SYN, SYN-ACK, ACK)")
    print("   The client can start sending data after the 3rd message.")

show_quiz_answers()

## 📚 Summary

### What We Learned:

1. **OSI Layers** - Networks are layered; we mostly care about L3 (IP), L4 (TCP/UDP), L7 (HTTP)

2. **TCP vs UDP**:
   - TCP: Reliable but slow setup (handshake)
   - UDP: Fast but unreliable (no guarantees)

3. **HTTP Lifecycle**: DNS → TCP Handshake → Request → Response → Teardown

4. **Load Balancers**:
   - L4: Fast, maintains connections, can't inspect content
   - L7: Smart routing, but breaks persistent connections

### Next Up: Simple Polling

In the next notebook, we'll build our first real-time update system using simple polling - the most straightforward approach!